# KQ Water Sales - 3/3 Order Anomaly Detection (Isolation Forest) - Google Colab
Capstone: Data-Driven Water Sales and Order Management System for Kenya Airways | Strathmore ICS 4D
Trains `analytics/ml/anomaly_detection.py` -> `detect_anomalies(order_queryset)`.
Run: Upload to Colab -> Runtime -> Run all (CPU only, 5 min). Output: `anomaly_isolationforest.pkl` + `anomalies.csv`.


## 0. Datasets — what to train on and why

Your KQ water-sales data does not exist publicly (it is internal B2B data), so we train on a **proxy public dataset with the exact same shape** (InvoiceNo ≈ Order, StockCode ≈ Product, Quantity/UnitPrice ≈ OrderItem, CustomerID ≈ Customer), then fine-tune on a **synthetic KQ water dataset** that matches your ERD.

### Recommended datasets

| Dataset | Rows | Why it fits | Link / access | Use for |
|---|---|---|---|---|
| **UCI Online Retail II (PRIMARY — used by default below)** | 1,067,371 txns, Dec 2009–Dec 2011 | Transactional B2B/wholesale sales, RFM case-study dataset. Columns map 1:1 to your `Order/OrderItem/Customer` | https://archive.ics.uci.edu/dataset/502/online+retail+ii — direct `.xlsx`: `https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx` | forecasting + RFM segmentation + anomaly |
| UCI Online Retail I | 541,909 txns, Dec 2010–Dec 2011 | Smaller subset of the above, faster download | https://archive.ics.uci.edu/dataset/352/online+retail | quick experiments |
| Olist Brazilian E-Commerce (alternative) | 100k orders, 2016–2018, 9 CSVs | Real orders + payments + customers, good for RFM | https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce (needs Kaggle login + API key) | segmentation / trend |
| **Synthetic KQ Water Sales (built into §2 below)** | you choose, default 2 years daily | Products: `500ml x24, 1L x12, 5L, 10L, 20L dispenser`; customers: catering units, lounges, offices, distributors; weekly seasonality + pay-day spikes — defensible in your Chapter 4 as domain-adapted data | no download, generated in-notebook | domain fine-tuning + final demo when real KQ data is unavailable |

**Schema mapping (proxy → your Django models):**

```
InvoiceNo      → orders.Order.order_id (grouped by InvoiceNo+InvoiceDate)
CustomerID     → orders.Customer.customer_id
StockCode/Description → inventory.Product.product_id / name
Quantity       → orders.OrderItem.quantity
UnitPrice      → orders.OrderItem.unit_price / inventory.Product.unit_price
InvoiceDate    → orders.Order.order_date
Quantity*UnitPrice → orders.Invoice.total_amount / Payment.amount
```

In [ ]:
# -- 1. Install (Colab) --
# Pinned to match requirements.txt in the Django repo.
!pip install -q scikit-learn==1.5.0 pandas==2.2.2 numpy==1.26.4 statsmodels==0.14.2 openpyxl==3.1.5 joblib matplotlib seaborn ucimlrepo kaggle
print("installs done")

In [ ]:
# ── 2. Imports ──────────────────────────────────────────────────────
import os, warnings, joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import IsolationForest
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
ART = Path("/content/ml_artifacts"); ART.mkdir(parents=True, exist_ok=True)
print("artifacts dir:", ART.resolve())

## 1. Load data

**Option A (default):** UCI Online Retail II via `ucimlrepo` (no login). Falls back to direct `.xlsx` URL, then to synthetic data if offline.  
**Option B:** synthetic KQ water-sales data — set `USE_SYNTHETIC_ONLY = True` to train purely on domain data (recommended for your final demo screenshots).

In [ ]:
DATASET = "UCI"  # options: "UCI" | "KQ_SYNTHETIC" | "OLIST"
SYNTHETIC_DAYS = 730          # used when DATASET == "KQ_SYNTHETIC"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def setup_kaggle():
    """Configure Kaggle credentials in Colab.

    Option A (recommended): Colab -> Secrets (key icon, left sidebar) -> add
      KAGGLE_USERNAME and KAGGLE_KEY from kaggle.com -> Settings -> API -> Create New Token.
      Then this function picks them up automatically via google.colab.userdata.
    Option B: upload your kaggle.json manually to /content/kaggle.json when prompted.
    """
    import os, json, shutil
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    dst = os.path.expanduser("~/.kaggle/kaggle.json")
    if os.path.exists(dst):
        print("kaggle.json already configured")
        return True
    # Try Colab Secrets
    try:
        from google.colab import userdata
        u = userdata.get("KAGGLE_USERNAME"); k = userdata.get("KAGGLE_KEY")
        if u and k:
            json.dump({"username": u, "key": k}, open(dst, "w"))
            os.chmod(dst, 0o600)
            print("Kaggle auth loaded from Colab Secrets")
            return True
    except Exception as e:
        print("Colab Secrets not found:", str(e)[:150])
    # Manual upload fallback
    try:
        from google.colab import files
        print("Upload your kaggle.json (Kaggle -> profile -> Settings -> API -> Create New Token)")
        up = files.upload()
        for name in up:
            if name.endswith(".json"):
                shutil.copy(f"/content/{name}", dst)
                os.chmod(dst, 0o600)
                print("kaggle.json configured from upload")
                return True
    except Exception as e:
        print("upload path unavailable (not in Colab?):", str(e)[:150])
    print("WARNING: no Kaggle credentials - OLIST will fail, use UCI or KQ_SYNTHETIC")
    return False

def load_uci_retail():
    """UCI Online Retail II (id=502) -> direct xlsx fallback. No login needed."""
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=502)
        df = pd.DataFrame(ds.data.features, columns=ds.variables.name.tolist()[:ds.data.features.shape[1]])
        full = ds.data.original if hasattr(ds.data, "original") else df
        print("loaded via ucimlrepo:", full.shape)
        return pd.DataFrame(full)
    except Exception as e:
        print("ucimlrepo failed:", type(e).__name__, str(e)[:200])
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx"
    print("trying direct xlsx ...")
    return pd.read_excel(url)  # ~1 min in Colab

def load_olist_from_kaggle():
    """Fetch olistbr/brazilian-ecommerce via Kaggle API, map 9 CSVs -> KQ schema.

    Mapping: order_id -> InvoiceNo, customer_unique_id -> CustomerID,
    product_id -> StockCode, price -> UnitPrice, purchase_timestamp -> InvoiceDate.
    Quantity is 1 per order-item row (Olist has no qty column) - aggregated per order.
    """
    import zipfile
    setup_kaggle()
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.dataset_download_files("olistbr/brazilian-ecommerce", path="/content/olist", unzip=True)
    base = Path("/content/olist")
    orders = pd.read_csv(base / "olist_orders_dataset.csv", parse_dates=["order_purchase_timestamp"])
    items = pd.read_csv(base / "olist_order_items_dataset.csv")
    products = pd.read_csv(base / "olist_products_dataset.csv")
    customers = pd.read_csv(base / "olist_customers_dataset.csv")
    cat = pd.read_csv(base / "product_category_name_translation.csv")
    products = products.merge(cat, on="product_category_name", how="left")
    cust_map = customers.set_index("customer_id")["customer_unique_id"].to_dict()
    df = items.merge(orders[["order_id", "customer_id", "order_purchase_timestamp"]], on="order_id", how="left")
    df["CustomerID"] = df["customer_id"].map(cust_map).fillna(df["customer_id"])
    df["InvoiceDate"] = pd.to_datetime(df["order_purchase_timestamp"])
    df["Quantity"] = 1
    df["UnitPrice"] = df["price"].astype(float)
    df["Country"] = "Brazil"
    df = df.rename(columns={"order_id": "InvoiceNo", "product_id": "StockCode",
                            "product_category_name_english": "Description"})
    df["Description"] = df["Description"].fillna(df["StockCode"])
    print("OLIST mapped:", df.shape, "| orders:", df["InvoiceNo"].nunique(),
          "| customers:", df["CustomerID"].nunique())
    return df[["InvoiceNo", "StockCode", "Description", "Quantity",
               "InvoiceDate", "UnitPrice", "CustomerID", "Country"]]

def make_synthetic_kq(days=SYNTHETIC_DAYS, seed=RANDOM_STATE):
    """Domain-adapted KQ water sales: weekly seasonality + payday spike + growth trend."""
    rng = np.random.default_rng(seed)
    products = [
        ("WTR-500x24", "Still Water 500ml x24", 520.0),
        ("WTR-1Lx12",  "Still Water 1L x12",  480.0),
        ("WTR-5L",     "Still Water 5L",      220.0),
        ("WTR-10L",    "Still Water 10L",     380.0),
        ("WTR-20L",    "Dispenser Water 20L", 450.0),
    ]
    customers = [(f"KQ-{i:03d}", n) for i, n in enumerate([
        "JKIA Catering Unit", "Jambojet Catering", "Pride Lounge", "Simba Lounge",
        "KQ HQ Offices", "Cargo Village", "KQ MRO Hangar", "Distributor-Nairobi",
        "Distributor-Mombasa", "Distributor-Kisumu", "Staff Shop", "Contractor-Cleaning",
    ], start=1)]
    dates = pd.date_range(end=pd.Timestamp.today().normalize(), periods=days, freq="D")
    rows = []
    inv = 10000
    for d in dates:
        dow, dom = d.dayofweek, d.day
        weekly = 1.35 if dow in (4, 5) else (0.55 if dow == 6 else 1.0)
        payday = 1.5 if dom in (27, 28, 29, 30, 1, 2) else 1.0
        trend = 1 + (d - dates[0]).days / days * 0.35
        n_orders = max(1, int(rng.poisson(14 * weekly * payday * trend)))
        for _ in range(n_orders):
            inv += 1
            cid, cname = customers[rng.integers(len(customers))]
            n_lines = rng.integers(1, 4)
            for _ in range(n_lines):
                code, desc, price = products[rng.integers(len(products))]
                qty = int(max(1, rng.normal(22 if "20L" in desc or "500" in desc else 10, 8)))
                if rng.random() < 0.03:
                    qty = int(qty * rng.choice([6, 8, 0.05]))
                rows.append([f"{inv}", code, desc, qty, d + pd.Timedelta(hours=int(rng.integers(6, 20))), price, cid, "Kenya"])
    return pd.DataFrame(rows, columns=["InvoiceNo","StockCode","Description","Quantity","InvoiceDate","UnitPrice","CustomerID","Country"])

if DATASET == "KQ_SYNTHETIC":
    raw = make_synthetic_kq()
    print("SYNTHETIC KQ dataset:", raw.shape)
elif DATASET == "OLIST":
    raw = load_olist_from_kaggle()
    print("OLIST dataset:", raw.shape)
else:  # UCI default
    try:
        raw = load_uci_retail()
        print("UCI dataset:", raw.shape)
    except Exception as e:
        print("UCI download failed, using synthetic KQ fallback:", str(e)[:300])
        raw = make_synthetic_kq()
raw.head(3)


In [ ]:
# ── 3. Clean + map to KQ schema ───────────────────────────────────────
df = raw.copy()
df.columns = [c.strip() for c in df.columns.astype(str)]
df = df.rename(columns={c: {"invoiceno":"InvoiceNo","stockcode":"StockCode","description":"Description","quantity":"Quantity","invoicedate":"InvoiceDate","unitprice":"UnitPrice","customerid":"CustomerID","country":"Country"}.get(c.lower(), c) for c in df.columns})
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df = df.dropna(subset=["InvoiceDate","CustomerID","Quantity","UnitPrice"])
df = df[(df["Quantity"] != 0)]  # keep negatives as returns/cancellations for anomaly model, drop zeros
df["CustomerID"] = df["CustomerID"].astype(str)
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
# cancellations in UCI start with 'C' — keep flag
df["is_cancel"] = df["InvoiceNo"].astype(str).str.startswith("C").astype(int)
print(df.shape, df["InvoiceDate"].min(), "→", df["InvoiceDate"].max())
print("customers:", df["CustomerID"].nunique(), "| products:", df["StockCode"].nunique())
df.head()

## 4. Model 3 — Anomaly detection (Isolation Forest → `detect_anomalies()`)

One row per **order** (InvoiceNo): quantity, revenue, #lines, avg unit price, cancel flag. Flags sudden spikes, dormant-account reactivation, and bulk returns for the dashboard's *ML Anomaly Detector* panel.

In [ ]:
# ── order-level features ──────────────────────────────────────────────
ord_f = df.groupby("InvoiceNo").agg(customer=("CustomerID","first"), date=("InvoiceDate","first"), qty=("Quantity","sum"), revenue=("Revenue","sum"), lines=("StockCode","count"), avg_price=("UnitPrice","mean"), canc=("is_cancel","max"))
ord_f["unit_qty_ratio"] = ord_f["qty"] / ord_f["lines"].replace(0, 1)
FEATS = ["qty","revenue","lines","avg_price","unit_qty_ratio"]
Xa = StandardScaler().fit_transform(np.log1p(ord_f[FEATS].clip(lower=0)))
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=RANDOM_STATE).fit(Xa)
ord_f["anomaly_score"] = -iso.score_samples(Xa)  # higher = weirder
ord_f["is_anomaly"] = (iso.predict(Xa) == -1).astype(int)
print("orders:", len(ord_f), "| anomalies:", int(ord_f.is_anomaly.sum()))
ord_f[ord_f.is_anomaly==1].sort_values("anomaly_score", ascending=False).head(10)

In [ ]:
# ── describe + save ───────────────────────────────────────────────────
def describe_row(r):
    bits = []
    if abs(r.qty) > ord_f.qty.quantile(.995): bits.append(f"extreme quantity {r.qty:.0f}")
    if r.revenue > ord_f.revenue.quantile(.995): bits.append(f"high value KES {r.revenue:,.0f}")
    if r.canc == 1: bits.append("cancellation/return")
    if r.lines > ord_f.lines.quantile(.99): bits.append(f"many lines ({r.lines:.0f})")
    return ("Order " + str(r.name) + " (" + str(r.customer) + ") — " + (", ".join(bits) if bits else "statistical outlier"))
anoms = ord_f[ord_f.is_anomaly==1].copy()
anoms["description"] = [describe_row(anoms.iloc[i]) for i in range(len(anoms))]
joblib.dump(iso, ART / "anomaly_isolationforest.pkl")
anoms.reset_index().to_csv(ART / "anomalies.csv", index=False)
print("saved anomaly_isolationforest.pkl")
plt.figure(figsize=(11,4)); plt.scatter(ord_f.revenue.clip(upper=ord_f.revenue.quantile(.99)), ord_f.qty.clip(-100, ord_f.qty.quantile(.99)), c=ord_f.is_anomaly, cmap="coolwarm", alpha=.5, s=8)
plt.xlabel("revenue"); plt.ylabel("qty"); plt.title("Isolation Forest anomalies (red) — revenue × quantity per order"); plt.show()
print(anoms["description"].head(5).to_string())

## Export
Download `anomaly_isolationforest.pkl` + `anomalies.csv` into `analytics/ml/artifacts/`. `Report.generate(Report.ReportType.SALES_TREND)` loads the pkl (see `anomaly_detection.py`).


In [ ]:
import shutil
from pathlib import Path
ART = Path("/content/ml_artifacts")
print("\n".join(sorted(str(p) for p in ART.iterdir())))
shutil.make_archive("/content/kq_anomaly_artifact", "zip", ART)
try:
    from google.colab import files; files.download("/content/kq_anomaly_artifact.zip")
except Exception:
    print("zip at /content/kq_anomaly_artifact.zip")
